# Inferring

In prompting, **inferring** means asking the model to derive information that isn't stated explicitly — 
extracting sentiment, intent, topics, labels, or structured data from unstructured text.

Unlike summarizing (which compresses what's there), inferring pulls out what's *implied*.

## What the examples below demonstrate

| Task | What we infer |
|---|---|
| Sentiment analysis | Is this review positive, negative, or mixed? |
| Emotion detection | What emotions does the writer express? |
| Topic extraction | What subjects does this text cover? |
| Structured extraction | Pull specific fields (price, name, issue) from free text |
| Multi-label inference | Infer several things from a single prompt |

## The prompting pattern

Your task is to [infer X] from the text below.
Text: {text}



Each cell below sends the same or similar text to Claude with a different inference goal.
The key insight: the *same input* produces very different outputs depending on what you ask for.


In [21]:
from anthropic import Anthropic
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
anthropic_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
SYSTEM_PROMPT = "You are a senior Java and Spring Boot engineer."
MODEL = "claude-sonnet-4-6"
MAX_TOKENS = 2048
response = anthropic_client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=[{"role": "user", "content": "Say hello in one sentence."}]
)
print(response.content[0].text)

Hello there! I hope you're having a wonderful day! 😊


In [22]:
def chat(messages, user_message):
    messages.append({"role": "user", "content": user_message})
    response = anthropic_client.messages.create(
        model=MODEL, max_tokens=MAX_TOKENS, system=SYSTEM_PROMPT, messages=messages
    )
    print(response.content[0].text)
    messages.append({"role": "assistant", "content": response.content[0].text})
    return response

# Sentiment (positive/negative)

In [19]:
messages = []
review = """In a Latenode Tech Blog review, Vasiliy Datsenko describes testing Grok AI, noting its impressive speed and deep, human-like contextual understanding. While praising its capabilities in complex tasks like code debugging and sarcasm detection, the review highlights that Grok's lack of content guardrails makes it feel like an "unfiltered internet native". The author concludes that while it is brilliant for individual power users, it poses risks for corporate environments due to its unpredictable nature."""

prompt = f"""
What is the sentiment of this review delimited by triple backticks? Answer with one word: positive, negative, or neutral.
```{review}```
"""
r1 = chat(messages, prompt)  

positive


# Identify types of emotions

In [20]:
prompt = f"""
Identify a list of emotions that the writer of the \
following review is expressing. Include no more than \
five items in the list. Format your answer as a list in \
JSON format.

Review text: ```{review}```
"""
r2 = chat(messages, prompt)


```json
[
  "admiration",
  "enthusiasm",
  "concern",
  "caution",
  "ambivalence"
]
```


# Identify anger

In [12]:
prompt = f"""
Is the writer of the following review expressing anger?\
The review is delimited with triple backticks. \
Give your answer as either yes or no.

Review text: '''{review}'''
"""
r2 = chat(messages, prompt)

No


# Inferring topics
# Topic Extraction

*Article source: BBC News*

Extract the main topics discussed in this article about the Russia-Ukraine conflict.


In [16]:
story ="""Russia carried out a deadly large-scale wave of strikes against Ukraine, firing hundreds of drones and dozens of missiles overnight.

Ukrainian President Volodymyr Zelensky said Kyiv was the main target, but other areas were also hit, with about 100 people injured.

Four people were killed in the capital and wider region, with loud explosions heard across the area throughout the night. Dozens of residential buildings, a school, an opera house and a museum were damaged.

Russia's defence ministry said the Oreshnik hypersonic missile was used in the strikes, which it described as coming in response to Ukraine's "attacks on civilian infrastructure". Ukraine's military denies targeting civilians.

Russian President Vladimir Putin launched a full-scale invasion of Ukraine in February 2022.

Earlier this week, he accused Kyiv of hitting a student dormitory in the town of Starobilsk on Friday, in which Russian officials said 21 people were killed.

Ukraine's military said its forces did carry out an attack in Starobilsk in Russian-occupied eastern Ukraine overnight on Friday, but maintained that it struck an elite Russian drone military unit.

European leaders have condemned the overnight Russian strikes on Sunday, which came after warnings from Zelensky that Russia was planning an attack, and that it may have been preparing to use the Oreshnik missile."""

In [23]:
# Infer Topics
prompt = f"""
Determine five topics that are being discussed in the \
following text, which is delimited by triple backticks.

Make each item one or two words long. 

Format your response as a list of items separated by commas.

Text sample: '''{story}'''
"""
r3 = chat(messages, prompt)

Russian strikes, Ukraine conflict, civilian casualties, hypersonic missile, European response
